In [130]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import geopandas as gpd

In [154]:
df_20182021 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2018_2021.csv", thousands=",")
df_20212026 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2021_2026.csv", thousands=",")

C:\Users\gelet\AppData\Local\Temp\ipykernel_23404\873818448.py:1: DtypeWarning: Columns (32: search_types, 41: legal_basises_relevant_factors, 42: subject_restrained, 43: subject_restraint_devices) have mixed types. Specify dtype option on import or set low_memory=False.
  df_20182021 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2018_2021.csv", thousands=",")


In [132]:
print(df_20182021.shape)
print(df_20212026.shape)

(148205, 47)
(86729, 47)


In [155]:
# Change time variable into datetime from string
df_20182021["EventDate"] = df_20182021["EventDate"].str.strip()
df_20182021["EventDate"] =  pd.to_datetime(df_20182021["EventDate"], format="%Y %b %d %I:%M:%S %p", errors="coerce")

df_20212026["EventDate"] = df_20212026["EventDate"].str.strip()
df_20212026["EventDate"] =  pd.to_datetime(df_20212026["EventDate"], format="%Y %b %d %I:%M:%S %p", errors="coerce")

In [213]:
# Combine dataframes
df = pd.concat([df_20182021, df_20212026], ignore_index=True)

In [214]:
# Add extra date variables
df["month"] = df["EventDate"].dt.month
df["year"] = df["EventDate"].dt.year
df["weekend"] = (df["EventDate"].dt.weekday >= 5).astype(int)

# Only examine stops on or after March 1, 2018
df = df[(df["month"] >= 3) & (df["year"] >= 2018)]

# Only examine vehicle stops
df = df[(df["subject_type"] == "Passenger") | (df["subject_type"] == "Driver")]

In [215]:
# Remapping some categorical variables to 0, 1
df["SubjectGender"] = df["SubjectGender"].map({"FEMALE": 1, "MALE": 0})
df["SubjectHasPhotoID"] = df["SubjectHasPhotoID"].map({"Yes": 1, "No": 0})
df["isDriver"] = df["subject_type"].map({"Driver":1, "Passenger":0})
df["search_occurred"] = df["search_occurred"].map({"Yes": 1, "No": 0})
# Creating an indicator variable for whether or not a driver license state is LA
df["isLA"] = (df["SubjectDriverLicState"] == "LA").astype("Int64")
df.loc[df["SubjectDriverLicState"].isna(), "isLA"] = np.nan
# Remapping zip code as a categorical instead of numerical variable
df["Zip"] = df["Zip"].astype("Int64").astype(str)

In [216]:
# Dropping post search variables
# We cannot use these variables as predictors because they may/will only happen after a search already occurs
df = df.drop(["evidence_seized", "search_types", "consent_to_search", "pat_down", "consent_form_completed", "stripbody_cavity_search", "legal_basises", "legal_basises_relevant_factors", "stop_results", "subject_restrained", "subject_restraint_devices", "exit_vehicle", "evidence_types"], axis=1)

# Dropping redundant variables
df = df.drop(["EventDate", "ActionsTaken", "CreatedDateTime", "LastModifiedDateTime", "subject_type"], axis=1)

In [217]:
# Potential outlier columns: VehicleYear, SubjectAge, SubjectHeight
# Impute impossible or highly implausible values as NaN
df["VehicleYear"] = np.where((df["VehicleYear"] == 1900) | (df["VehicleYear"] >= 2028), np.nan, df["VehicleYear"])

df["SubjectAge"] = np.where((df["SubjectAge"] < 0) | (df["SubjectAge"] > 119), np.nan, df["SubjectAge"])

df["SubjectHeight"] = np.where((df["SubjectHeight"] <= 20) | (df["SubjectHeight"] >= 100), np.nan, df["SubjectHeight"])

df["SubjectWeight"] = np.where((df["SubjectWeight"] <= 10) | (df["SubjectWeight"] >= 800), np.nan, df["SubjectWeight"])

In [218]:
# Introduce BMI variable and check which are outliers
df["orig_height"] = df["SubjectHeight"]
df["orig_weight"] = df["SubjectWeight"]
df["BMI"] = 703 * (df["SubjectWeight"] / df["SubjectHeight"]**2)

# We flag rows with a BMI less than or more than
# https://pmc.ncbi.nlm.nih.gov/articles/PMC8943548/
df["vital_flag"] = ((df["BMI"] < 11) | (df["BMI"] > 70)).astype(int)
print(len(df[df["vital_flag"] == 1])) # 285

# If BMI < 11, we suspect a decimal shift error:
# Multiply weight by 10
# Use a sharper lower cutoff
df["SubjectWeight"] = np.where((df["vital_flag"] == 1) & (df["BMI"] < 9),
                               df["SubjectWeight"]*10, df["SubjectWeight"])

# If BMI > 70, height is probably misinputed
# Set as nan if height is less than 40
df["SubjectHeight"] = np.where((df["vital_flag"] == 1) & (df["BMI"] > 70) & (df["SubjectHeight"] < 40), np.nan, df["SubjectHeight"])

285


In [219]:
df

,FieldInterviewID,NOPD_Item,District,Zone,OfficerAssignment,StopDescription,VehicleYear,VehicleMake,VehicleModel,VehicleStyle,...,night,month,year,weekend,isDriver,isLA,orig_height,orig_weight,BMI,vital_flag
1,571395,J0050921,4,E,4th District,TRAFFIC VIOLATION,2009.0,HONDA,ODYSSEY,FOUR DOOR,...,1,10,2021,0,1,1,66.0,129.0,20.818871,0
2,571394,J0115921,7,F,7th District,TRAFFIC VIOLATION,2009.0,LEXUS,OTHER,FOUR DOOR,...,1,10,2021,0,1,<NA>,67.0,220.0,34.453108,0
3,571393,J0112821,3,J,ISB,TRAFFIC VIOLATION,2011.0,FORD,FOCUS,FOUR DOOR,...,1,10,2021,0,1,1,71.0,185.0,25.799445,0
4,571391,J0110421,7,O,7th District,TRAFFIC VIOLATION,2018.0,MERCEDES-BENZ,OTHER,FOUR DOOR,...,1,10,2021,0,1,1,67.0,170.0,26.622856,0
5,571392,J0104121,7,F,7th District,TRAFFIC VIOLATION,2013.0,DODGE,CHARGER,FOUR DOOR,...,1,10,2021,0,1,1,67.0,140.0,21.924705,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234919,571408,J0167721,4,J,Traffic,TRAFFIC VIOLATION,2010.0,CADILLAC,CTS,FOUR DOOR,...,0,10,2021,1,1,1,66.0,145.0,23.401056,0
234920,572608,J3007721,4,B,Traffic,TRAFFIC VIOLATION,2004.0,TOYOTA,TACOMA,PICK UP,...,0,10,2021,1,1,<NA>,69.0,200.0,29.531611,0
234922,571402,J0150921,5,C,5th District,TRAFFIC VIOLATION,2004.0,CHEVROLET,SILVERADO,PICK UP,...,1,10,2021,1,1,1,69.0,230.0,33.961353,0
234924,571401,J0145621,5,B,5th District,CALL FOR SERVICE,NaN,NaN,NaN,NaN,...,1,10,2021,1,1,0,72.0,210.0,28.478009,0


In [220]:
# A lot of ages clustered around 0 have misinputed age values, we work on fixing that now
# If the age is young enough ~(<7), but the height is above 55 (more than 95th percentile for boys at 7) https://www.cdc.gov/growthcharts/data/set1clinical/cj41c021.pdf, set their age as NA because their age is not correctly inputted
df["orig_age"] = df["SubjectAge"]
df["SubjectAge"] = np.where((df["SubjectHeight"] > 55) & (df["SubjectAge"] <= 7), np.nan, df["SubjectAge"])

In [225]:
# Dropping driver license state variable, officerassignment variable, stopdescription, VehicleModel and VehicleMake variable
# These variables are either too sparse (driver license state, model/make), seem to have no relevance (officer assignment) or have too many similar values (stop description)
df = df.drop(["SubjectDriverLicState", "OfficerAssignment", "NOPD_Item", "VehicleModel", "VehicleMake", "BlockAddress"], axis=1)

KeyError: "['SubjectDriverLicState', 'OfficerAssignment', 'NOPD_Item', 'VehicleModel', 'VehicleMake', 'BlockAddress'] not found in axis"

In [226]:
# Splitting data into test and train
# stratify is used to ensure the proportion of searches is the same in the test and train data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=10, stratify=df["search_occurred"])

In [229]:
test_df["search_occurred"].value_counts(normalize=True)

search_occurred
0    0.944939
1    0.055061
Name: proportion, dtype: float64

In [230]:
train_df["search_occurred"].value_counts(normalize=True)

search_occurred
0    0.944929
1    0.055071
Name: proportion, dtype: float64

In [231]:
train_df.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index=False)
test_df.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_test1.csv", index=False)